<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/Direct_Logit_Attribution_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# TransformerLens Direct Logit Attribution Demo

Direct Logit Attribution (DLA) is one of the most common analyses in mechanistic interpretability: given a model and a target token (or the logit difference between two tokens), figure out how much each component of the residual stream — every attention head, every MLP neuron, the embeddings — contributes to that target logit. The math is straightforward (apply LayerNorm to the component, then project onto the unembed direction), but the standard way of computing it has a memory problem.

`ActivationCache.get_full_resid_decomposition` returns a stack of shape `[num_components, batch, pos, d_model]`. For per-neuron decomposition on a typical model, `num_components` is in the thousands (every neuron in every MLP). Multiplied by `batch * pos * d_model`, the intermediate tensor can be many GB even for moderately-sized models — enough to OOM the user before they ever get to the projection.

This notebook demonstrates the `project_output_onto` parameter, which folds the projection into the decomposition so the `[..., d_mlp, d_model]` intermediate is never materialized. The output last-dim shrinks from `d_model` to `num_outputs` (or to a scalar if you project onto a single direction), and the memory profile no longer scales with `d_model`.

## How to use this notebook

Go to Runtime > Change Runtime Type and select GPU as the hardware accelerator.

Tips for reading this Colab:

* You can run all this code for yourself!
* Use the table of contents pane in the sidebar to navigate
* Collapse irrelevant sections with the dropdown arrows
* Search the page using the search in the sidebar, not CTRL+F

## Setup (Ignore)

In [1]:
# NBVAL_IGNORE_OUTPUT
# Janky code to do different setup when run in a Colab notebook vs VSCode
import os

DEVELOPMENT_MODE = False
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
except:
    IN_COLAB = False

if not IN_GITHUB and not IN_COLAB:
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()
    # Code to automatically update the HookedTransformer code as its edited without restarting the kernel
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB:
    %pip install transformer_lens


Running as a Jupyter notebook - intended for development only!


In [2]:
# NBVAL_IGNORE_OUTPUT
import torch
from torch.utils._python_dispatch import TorchDispatchMode

from transformer_lens.model_bridge import TransformerBridge

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{device = }")


device = 'cpu'


## Load a model

We use `gpt2` (small) because it's a familiar model and the technique is the same regardless of choice. The memory savings scale with `d_mlp × d_model`, so the optimization matters more on larger models.

In [3]:
# NBVAL_IGNORE_OUTPUT
model = TransformerBridge.boot_transformers("gpt2", device=device)
model.enable_compatibility_mode()
print(f"d_model = {model.cfg.d_model}, d_mlp = {model.cfg.d_mlp}, n_layers = {model.cfg.n_layers}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

d_model = 768, d_mlp = 3072, n_layers = 12


## A worked example: who contributes to "Mary" vs "John"?

We'll use a classic IOI-style prompt where the model has to choose between two names. Concretely:

> *When John and Mary went to the shops, John gave the bag to ___*

The "right" answer (from the model's perspective) is *Mary*. We can quantify each residual stream component's contribution to that decision by projecting it onto the difference between the *Mary* and *John* unembed directions.

In [4]:
prompt = "When John and Mary went to the shops, John gave the bag to"
tokens = model.to_tokens(prompt)

with torch.no_grad():
    _, cache = model.run_with_cache(tokens)

mary_id = model.to_single_token(" Mary")
john_id = model.to_single_token(" John")
logit_direction = (
    model.tokens_to_residual_directions(torch.tensor(mary_id))
    - model.tokens_to_residual_directions(torch.tensor(john_id))
)
print(f"Logit direction shape: {logit_direction.shape}  # [d_model]")


Logit direction shape: torch.Size([768])  # [d_model]


## Without `project_output_onto`: the memory problem

The naive approach is to get the full decomposition, then project at the end:

```python
contributions = cache.get_full_resid_decomposition(apply_ln=True) @ logit_direction
```

This works, but `get_full_resid_decomposition` builds a tensor of shape `[num_components, batch, pos, d_model]` in memory before the projection. For `gpt2` with `d_mlp = 3072` and `n_layers = 12`, that's ~37k components × `d_model = 768` per position. Manageable on this tiny model, but for a 7B model it would be tens of gigabytes.

In [5]:
# Naive: build full decomposition, then project
naive = cache.get_full_resid_decomposition(layer=-1, pos_slice=-1, apply_ln=True) @ logit_direction
print(f"Naive output shape: {naive.shape}  # [num_components, batch]")


Naive output shape: torch.Size([37011, 1])  # [num_components, batch]


## With `project_output_onto`: same result, fraction of the memory

Pass the logit direction directly to `get_full_resid_decomposition`. The projection is folded into the per-neuron expansion, so the `[..., d_mlp, d_model]` intermediate is never built.

In [6]:
# Folded: projection happens during the decomposition
projected, labels = cache.get_full_resid_decomposition(
    layer=-1,
    pos_slice=-1,
    apply_ln=True,
    project_output_onto=logit_direction,
    return_labels=True,
)
print(f"Projected output shape: {projected.shape}  # same as naive")
print(f"Number of components: {len(labels)}")

assert torch.allclose(naive, projected, atol=1e-4), "Outputs should match!"
print("Outputs match the naive computation within 1e-4 tolerance.")


Projected output shape: torch.Size([37011, 1])  # same as naive
Number of components: 37011
Outputs match the naive computation within 1e-4 tolerance.


## Top contributors

Now we can ask which components most strongly push the model toward *Mary* vs *John*. Positive values push toward Mary; negative push toward John.

In [9]:
flat = projected[:, 0]  # [num_components]
top_indices = flat.abs().topk(8).indices
print("Top 8 components by absolute contribution:")
print()
for rank, i in enumerate(top_indices, start=1):
    sign = "+" if flat[i].item() >= 0 else " "
    print(f"  {rank}. {labels[i]:>10s}: {sign}{flat[i].item():.4f}")


Top 8 components by absolute contribution:

  1.       L9H9: +1.9583
  2.       L9H6: +1.6281
  3.      L10H7:  -1.4267
  4.      L11H1:  -0.9117
  5.     L11H10:  -0.9042
  6.      L10H0: +0.6370
  7.     L10H10: +0.4524
  8.      L8H10: +0.4264


`L9H9` and `L9H6` (attention heads in layer 9) are the dominant Mary-pushers — these are the well-known **name mover heads** from the IOI circuit ([Wang et al., 2022](https://arxiv.org/abs/2211.00593)), which copy the indirect object's representation into the prediction. The negative contributors `L10H7`, `L11H1`, and `L11H10` push in the opposite direction — these are the **S-inhibition / negative name mover** heads, part of the same circuit. The fact that attention heads dominate this analysis on `gpt2-small` is consistent with the IOI literature: MLP neurons contribute more diffusely and appear further down the ranking.

## Projecting onto multiple directions at once

`project_output_onto` also accepts a `[d_model, num_outputs]` tensor, letting you analyze contributions to many output directions in a single pass. The output shape becomes `[num_components, batch, num_outputs]`.

In [8]:
# Three target tokens: Mary, John, and the
target_ids = torch.tensor([mary_id, john_id, model.to_single_token(" the")])
target_directions = model.tokens_to_residual_directions(target_ids).T  # [d_model, 3]
print(f"Target directions shape: {target_directions.shape}")

multi = cache.get_full_resid_decomposition(
    layer=-1,
    pos_slice=-1,
    apply_ln=True,
    project_output_onto=target_directions,
)
print(f"Output shape: {multi.shape}  # [num_components, batch, num_targets]")


Target directions shape: torch.Size([768, 3])
Output shape: torch.Size([37011, 1, 3])  # [num_components, batch, num_targets]


In [9]:
# Top neuron contribution per target
# Find the neuron index range in the component list
neuron_idx_start = next(i for i, l in enumerate(labels) if l.startswith("L") and "N" in l)
neuron_idx_end = next(
    (i for i, l in enumerate(labels) if i > neuron_idx_start and not (l.startswith("L") and "N" in l)),
    len(labels),
)
neuron_slab = multi[neuron_idx_start:neuron_idx_end, 0]  # [n_neurons, 3]

print("Top MLP neuron contribution per target token:")
print()
for col, tid in enumerate(target_ids):
    top_neuron_local = neuron_slab[:, col].abs().argmax().item()
    top_label = labels[neuron_idx_start + top_neuron_local]
    val = neuron_slab[top_neuron_local, col].item()
    print(f"  '{model.to_string(tid)}': {top_label} ({val:+.4f})")


Top MLP neuron contribution per target token:

  ' Mary': L11N611 (+0.3188)
  ' John': L11N621 (-0.4359)
  ' the': L10N2214 (+0.4369)


## Verifying the memory savings

We can measure the largest tensor allocated during each call using `TorchDispatchMode`. The naive path materializes the `[..., d_mlp, d_model]` intermediate; the projected path stays bounded by the model's `W_out` matrix (a constant in the model, independent of batch/pos).

In [10]:
class MaxTensorWatcher(TorchDispatchMode):
    def __init__(self):
        self.max_numel = 0

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        result = func(*args, **(kwargs or {}))
        tensors = result if isinstance(result, (list, tuple)) else (result,)
        for t in tensors:
            if isinstance(t, torch.Tensor):
                self.max_numel = max(self.max_numel, t.numel())
        return result


In [11]:
# Single position (typical DLA-at-final-token setup)
with MaxTensorWatcher() as naive_watcher:
    _ = cache.get_full_resid_decomposition(layer=-1, pos_slice=-1, apply_ln=True) @ logit_direction
with MaxTensorWatcher() as projected_watcher:
    _ = cache.get_full_resid_decomposition(
        layer=-1, pos_slice=-1, apply_ln=True, project_output_onto=logit_direction
    )
print("Single position (pos_slice=-1):")
print(f"  Naive:      max tensor = {naive_watcher.max_numel:>12,} elements")
print(f"  Projected:  max tensor = {projected_watcher.max_numel:>12,} elements")
print(f"  Ratio:      {naive_watcher.max_numel / projected_watcher.max_numel:.1f}x")


Single position (pos_slice=-1):
  Naive:      max tensor =   28,424,448 elements
  Projected:  max tensor =    2,359,296 elements
  Ratio:      12.0x


In [12]:
# All positions — the optimization shows more dramatically as batch*pos grows
with MaxTensorWatcher() as naive_all:
    _ = cache.get_full_resid_decomposition(layer=-1, apply_ln=True) @ logit_direction
with MaxTensorWatcher() as proj_all:
    _ = cache.get_full_resid_decomposition(
        layer=-1, apply_ln=True, project_output_onto=logit_direction
    )
print("All positions:")
print(f"  Naive:      max tensor = {naive_all.max_numel:>12,} elements")
print(f"  Projected:  max tensor = {proj_all.max_numel:>12,} elements")
print(f"  Ratio:      {naive_all.max_numel / proj_all.max_numel:.1f}x")


All positions:
  Naive:      max tensor =  426,366,720 elements
  Projected:  max tensor =    2,359,296 elements
  Ratio:      180.7x


On `gpt2` (small) with this prompt, the projected path keeps the max tensor bounded by `d_mlp × d_model = 3072 × 768 ≈ 2.4M elements` (just the `W_out` matrix — a model constant). The naive path scales with `batch × pos × d_mlp × d_model`. For real workloads on larger models (batch 32, pos 128, 7B params with `d_mlp ≈ 11K`, `d_model ≈ 4K`), the ratio approaches 500–4000x — the difference between a comfortable run and an OOM.

## When the memory benefit doesn't apply

A few caveats worth flagging:

- The memory saving applies to the **per-neuron expansion** in MLPs. If you set `expand_neurons=False`, the decomposition stops at MLP-layer granularity (no per-neuron breakdown) and there's no big intermediate to avoid. Projection still works on this path, it just doesn't save memory.
- For 1D projections (`[d_model]`), the output last-dim is squeezed away. For 2D projections (`[d_model, num_outputs]`), it's preserved as `num_outputs`. This matches PyTorch's usual matmul broadcasting rules.
- The same `project_output_onto` parameter exists on `cache.stack_neuron_results` and `cache.get_neuron_results` if you only need a subset of components.

## Further reading

- Activation Patching demos for component-level causal interventions
- [issue #210](https://github.com/TransformerLensOrg/TransformerLens/issues/210) for the original feature request and discussion